## Setting up the molecular Hamiltonian

In [1]:
# Force the local gqcpy to be imported
import sys
sys.path.insert(0, '../../build/gqcpy/')

import gqcpy

In [2]:
molecule = gqcpy.Molecule.ReadXYZ("../../gqcp/tests/data/h2_szabo.xyz" , 0)  # create a neutral molecule
N = molecule.numberOfElectrons()

In [3]:
spinor_basis = gqcpy.RSpinOrbitalBasis_d(molecule, "STO-3G")

In [4]:
S = spinor_basis.quantize(gqcpy.OverlapOperator())
print(S.parameters())

[[0.99999999 0.65931816]
 [0.65931816 0.99999999]]


In [5]:
sq_hamiltonian = spinor_basis.quantize(gqcpy.FQMolecularHamiltonian(molecule))  # 'sq' for 'second-quantized'

In [6]:
print(sq_hamiltonian.core().parameters())

[[-1.12040896 -0.95837989]
 [-0.95837989 -1.12040896]]


In [7]:
print(sq_hamiltonian.twoElectron().parameters())

[[[[0.77460593 0.44410762]
   [0.44410762 0.56967589]]

  [[0.44410762 0.2970285 ]
   [0.2970285  0.44410762]]]


 [[[0.44410762 0.2970285 ]
   [0.2970285  0.44410762]]

  [[0.56967589 0.44410762]
   [0.44410762 0.77460593]]]]


In [8]:
environment = gqcpy.RHFSCFEnvironment_d.WithCoreGuess(N, sq_hamiltonian, S)
solver = gqcpy.RHFSCFSolver_d.DIIS()

In [9]:
objective = gqcpy.DiagonalRHFFockMatrixObjective_d(sq_hamiltonian)  # use the default threshold of 1.0e-08

In [10]:
rhf_parameters = gqcpy.RHF_d.optimize(objective, solver, environment).groundStateParameters()

In [11]:
C = rhf_parameters.expansion()
print(C.matrix())

[[-0.54893405 -1.21146402]
 [-0.54893405  1.21146402]]


In [12]:
D = rhf_parameters.calculateScalarBasis1DM()
print(D.matrix())

[[0.60265718 0.60265718]
 [0.60265718 0.60265718]]


In [14]:
F = rhf_parameters.calculateScalarBasisFockMatrix(D, sq_hamiltonian)
print(F.parameters())

[[-0.36553732 -0.59388534]
 [-0.59388534 -0.36553732]]


In [15]:
J = rhf_parameters.calculateScalarBasisDirectMatrix(D, sq_hamiltonian)
print(J.parameters())

[[1.34543038 0.89330201]
 [0.89330201 1.34543038]]


In [16]:
K = rhf_parameters.calculateScalarBasisExchangeMatrix(D, sq_hamiltonian)
print(K.parameters())

[[1.18111747 1.05761492]
 [1.05761492 1.18111747]]
